84(10) = 1010100(2)

00 - безадресные команды

1 - архитектура фон Неймана

0 - сумма элементов массива

In [17]:
# ===== Emulator of 16-bit RISC von Neumann CPU with Assembler + ipywidgets GUI (for Google Colab) =====
# Paste into a single Colab cell and run.

from dataclasses import dataclass
import re
import time
import threading

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError:
    raise RuntimeError("Run this in Google Colab (ipywidgets required).")

# ---------------- ISA ----------------
# 16-bit instruction format:
# [15..11 OPCODE][10..8 rD][7..6 MODE][5..0 OPERAND]
#
# MODE:
# 00 IMM  : operand -> signed 6-bit immediate (-32..31)
# 01 DIR  : operand -> direct memory address (0..63)  (we also support @0..255 via pseudo-encoding; see assembler)
# 10 REG  : operand low3 -> src register
# 11 RIND : [Rsrc] where src=operand low3 (no offset)

# Opcodes (5-bit)
OP = {
    "NOP": 0,
    "LD":  1,
    "ST":  2,
    "ADD": 3,
    "ADC": 4,
    "SUB": 5,
    "SBC": 6,
    "MUL": 7,
    "CMP": 8,
    "INC": 9,
    "DEC": 10,
    "JMP": 11,
    "JZ":  12,
    "JNZ": 13,
    "JC":  14,
    "JNC": 15,
    "HLT": 31
}

MODE_IMM  = 0
MODE_DIR  = 1
MODE_REG  = 2
MODE_RIND = 3

REG_NAMES = {f"R{i}": i for i in range(6)}
REG_NAMES["ACC"] = 0  # alias

def u16(x): return x & 0xFFFF

def sign6(x):
    x &= 0x3F
    return x - 64 if x & 0x20 else x

@dataclass
class DecodedIR:
    word: int
    opcode: int
    rd: int
    mode: int
    operand: int

# ---------------- CPU ----------------
class CPU:
    def __init__(self, mem_words=256):
        self.mem_words = mem_words
        self.reset()

    def reset(self):
        self.R = [0]*6
        self.PC = 0
        self.IR = 0
        self.Z = 0
        self.N = 0
        self.C = 0
        self.V = 0
        self.halted = False
        self.RAM = [0]*self.mem_words
        self.last_asm = ""
        self.last_decoded = None
        self.last_fields = {}

    def load_words(self, origin, words):
        for i,w in enumerate(words):
            addr = origin + i
            if not (0 <= addr < self.mem_words):
                raise ValueError(f"Program too big / address out of RAM: {addr}")
            self.RAM[addr] = u16(w)

    def fetch_decode(self):
        self.IR = self.RAM[self.PC]
        self.PC = (self.PC + 1) & 0xFF
        w = self.IR
        opcode = (w >> 11) & 0x1F
        rd     = (w >> 8) & 0x07
        mode   = (w >> 6) & 0x03
        operand= w & 0x3F
        self.last_decoded = DecodedIR(w, opcode, rd, mode, operand)
        return self.last_decoded

    def set_flags_arith(self, result, carry=None, overflow=None):
        r = u16(result)
        self.Z = 1 if r == 0 else 0
        self.N = 1 if (r & 0x8000) else 0
        if carry is not None: self.C = 1 if carry else 0
        if overflow is not None: self.V = 1 if overflow else 0

    def read_op(self, mode, operand):
        """Return (value, effective_address or None)."""
        if mode == MODE_IMM:
            return u16(sign6(operand)), None
        elif mode == MODE_DIR:
            addr = operand  # 0..63
            return self.RAM[addr], addr
        elif mode == MODE_REG:
            src = operand & 0x07
            return self.R[src], None
        elif mode == MODE_RIND:
            src = operand & 0x07
            addr = self.R[src] & 0xFF
            return self.RAM[addr], addr
        else:
            raise ValueError("Bad addressing mode")

    def write_mem(self, mode, operand, value):
        if mode == MODE_DIR:
            addr = operand
        elif mode == MODE_RIND:
            src = operand & 0x07
            addr = self.R[src] & 0xFF
        else:
            raise ValueError("ST supports only DIR or RIND")
        self.RAM[addr] = u16(value)
        return addr

    def step(self, disasm_line=""):
        if self.halted:
            return

        d = self.fetch_decode()
        self.last_asm = disasm_line

        op = d.opcode
        rd = d.rd & 0x07
        mode = d.mode
        operand = d.operand

        # Safety: we only have R0..R5. If rd is 6..7 in machine code -> treat as 0 (or raise).
        if rd >= 6: rd = 0

        # Execute
        if op == OP["NOP"]:
            pass

        elif op == OP["HLT"]:
            self.halted = True

        elif op == OP["LD"]:
            val, ea = self.read_op(mode, operand)
            self.R[rd] = u16(val)
            self.set_flags_arith(self.R[rd])

        elif op == OP["ST"]:
            addr = self.write_mem(mode, operand, self.R[rd])
            # ST typically doesn't change flags; keep as is.

        elif op in (OP["ADD"], OP["ADC"], OP["SUB"], OP["SBC"], OP["MUL"], OP["CMP"]):
            b, _ = self.read_op(mode, operand)
            a = self.R[rd]

            if op == OP["ADD"]:
                full = a + b
                res = full
                carry = full > 0xFFFF
                self.R[rd] = u16(res)
                self.set_flags_arith(res, carry=carry)

            elif op == OP["ADC"]:
                full = a + b + (1 if self.C else 0)
                res = full
                carry = full > 0xFFFF
                self.R[rd] = u16(res)
                self.set_flags_arith(res, carry=carry)

            elif op == OP["SUB"]:
                full = (a - b)
                # Borrow model: C=1 means no borrow (a>=b)
                carry = a >= b
                self.R[rd] = u16(full)
                self.set_flags_arith(full, carry=carry)

            elif op == OP["SBC"]:
                borrow = 0 if self.C else 1
                full = a - b - borrow
                carry = (a - borrow) >= b
                self.R[rd] = u16(full)
                self.set_flags_arith(full, carry=carry)

            elif op == OP["MUL"]:
                full = (a * b)
                self.R[rd] = u16(full)
                # For MUL: set Z/N, and C if high part != 0
                carry = (full >> 16) != 0
                self.set_flags_arith(full, carry=carry)

            elif op == OP["CMP"]:
                full = a - b
                carry = a >= b
                self.set_flags_arith(full, carry=carry)

        elif op in (OP["INC"], OP["DEC"]):
            if op == OP["INC"]:
                res = self.R[rd] + 1
                self.R[rd] = u16(res)
                self.set_flags_arith(res, carry=(res > 0xFFFF))
            else:
                a = self.R[rd]
                res = a - 1
                self.R[rd] = u16(res)
                self.set_flags_arith(res, carry=(a >= 1))

        elif op in (OP["JMP"], OP["JZ"], OP["JNZ"], OP["JC"], OP["JNC"]):
            # Jumps use operand as DIR address (0..63) in machine code.
            # We additionally support full @0..255 via assembler "pseudo": it emits a special encoding:
            # If rd==7 then treat addr as (rd<<6)|operand. But rd is 3 bits; we used rd field.
            # To keep emulator simple: we use rd as upper 2 bits when opcode is jump:
            # addr = (rd << 6) | operand  -> 0..511, but PC is 8-bit => 0..255, so keep it 0..255.
            addr = ((d.rd & 0x03) << 6) | operand  # rd low2 bits as high bits for jump target
            addr &= 0xFF

            take = False
            if op == OP["JMP"]: take = True
            elif op == OP["JZ"]: take = (self.Z == 1)
            elif op == OP["JNZ"]: take = (self.Z == 0)
            elif op == OP["JC"]: take = (self.C == 1)
            elif op == OP["JNC"]: take = (self.C == 0)

            if take:
                self.PC = addr

        else:
            raise ValueError(f"Unknown opcode in IR: {op}")

# ---------------- Assembler ----------------
class AssemblerError(Exception):
    pass

class Assembler:
    token_re = re.compile(r"[,\s]+")

    def __init__(self):
        self.labels = {}
        self.origin = 0
        self.lines = []
        self.line_to_addr = {}

    def strip_comment(self, s):
        return s.split(";", 1)[0].strip()

    def parse_number(self, s):
        s = s.strip()
        if s.startswith("0x") or s.startswith("0X"):
            return int(s, 16)
        if s.startswith("0b") or s.startswith("0B"):
            return int(s, 2)
        return int(s, 10)

    def is_label_def(self, line):
        return bool(re.match(r"^[A-Za-z_]\w*:\s*$", line))

    def parse_operand(self, op_str):
        op_str = op_str.strip()
        # immediate
        if op_str.startswith("#"):
            return ("IMM", self.parse_number(op_str[1:]))
        # indirect [Rk]
        m = re.match(r"^\[(ACC|R[0-5])\]$", op_str, re.IGNORECASE)
        if m:
            r = m.group(1).upper()
            return ("RIND", REG_NAMES[r])
        # register
        m = re.match(r"^(ACC|R[0-5])$", op_str, re.IGNORECASE)
        if m:
            r = m.group(1).upper()
            return ("REG", REG_NAMES[r])
        # direct memory by @num
        if op_str.startswith("@"):
            return ("DIRA", self.parse_number(op_str[1:]))
        # label direct
        if re.match(r"^[A-Za-z_]\w*$", op_str):
            return ("LABEL", op_str)
        raise AssemblerError(f"Bad operand: {op_str}")

    def encode(self, opcode, rd, mode, operand6):
        return ((opcode & 0x1F) << 11) | ((rd & 0x07) << 8) | ((mode & 0x03) << 6) | (operand6 & 0x3F)

    def assemble(self, text):
        self.labels = {}
        self.origin = 0
        self.lines = []
        self.line_to_addr = {}

        raw_lines = text.splitlines()

        # Preprocess (keep for pass2)
        cleaned = []
        for i,ln in enumerate(raw_lines, start=1):
            s = self.strip_comment(ln)
            cleaned.append((i, s))
        self.lines = cleaned

        # Pass1: label addresses
        addr = 0
        for lineno, s in self.lines:
            if not s:
                continue
            # label+rest in one line: "LOOP: ADD R1, #1"
            if ":" in s:
                # split first label
                m = re.match(r"^([A-Za-z_]\w*):\s*(.*)$", s)
                if m:
                    lab = m.group(1)
                    rest = m.group(2).strip()
                    if lab in self.labels:
                        raise AssemblerError(f"Line {lineno}: duplicate label {lab}")
                    self.labels[lab] = addr
                    s = rest
                    if not s:
                        continue

            if s.upper().startswith(".ORG"):
                parts = self.token_re.split(s, maxsplit=1)
                if len(parts) < 2:
                    raise AssemblerError(f"Line {lineno}: .ORG requires address")
                addr = self.parse_number(parts[1])
                if not (0 <= addr < 256):
                    raise AssemblerError(f"Line {lineno}: .ORG out of range 0..255")
                continue

            if s.upper().startswith(".WORD"):
                # may contain many numbers separated by commas/spaces
                rest = s[5:].strip()
                if not rest:
                    raise AssemblerError(f"Line {lineno}: .WORD requires data")
                nums = [x for x in re.split(r"[,\s]+", rest) if x]
                addr += len(nums)
                continue

            # instruction = 1 word
            self.line_to_addr[lineno] = addr
            addr += 1

        # Pass2: encode
        addr = 0
        origin = 0
        words = []
        disasm_map = {}  # addr -> asm line (normalized)
        for lineno, s in self.lines:
            if not s:
                continue

            # handle "LABEL: rest"
            if ":" in s:
                m = re.match(r"^([A-Za-z_]\w*):\s*(.*)$", s)
                if m:
                    s = m.group(2).strip()
                    if not s:
                        continue

            if s.upper().startswith(".ORG"):
                parts = self.token_re.split(s, maxsplit=1)
                addr = self.parse_number(parts[1])
                origin = addr if not words else origin
                continue

            if s.upper().startswith(".WORD"):
                rest = s[5:].strip()
                nums = [x for x in re.split(r"[,\s]+", rest) if x]
                for n in nums:
                    words.append(u16(self.parse_number(n)))
                    disasm_map[addr] = f".WORD {n}"
                    addr += 1
                continue

            # instruction parse
            parts = [p for p in self.token_re.split(s.strip()) if p]
            mnem = parts[0].upper()
            if mnem not in OP:
                raise AssemblerError(f"Line {lineno}: unknown mnemonic {mnem}")

            opcode = OP[mnem]

            # formats:
            # - HLT / NOP: no operands
            # - INC/DEC: 1 reg
            # - JMP/JZ/JNZ/JC/JNC: 1 target (@addr or label)
            # - LD/ADD/.../CMP: "MN Rdest, op"
            # - ST: "ST Rsrc, @addr|[Rk]|label"
            if mnem in ("HLT","NOP"):
                w = self.encode(opcode, 0, MODE_IMM, 0)
                words.append(w)
                disasm_map[addr] = mnem
                addr += 1
                continue

            if mnem in ("INC","DEC"):
                if len(parts) != 2:
                    raise AssemblerError(f"Line {lineno}: {mnem} needs 1 register")
                r = parts[1].upper()
                if r not in REG_NAMES or REG_NAMES[r] > 5:
                    raise AssemblerError(f"Line {lineno}: bad register {r}")
                w = self.encode(opcode, REG_NAMES[r], MODE_IMM, 0)
                words.append(w)
                disasm_map[addr] = f"{mnem} {r}"
                addr += 1
                continue

            if mnem in ("JMP","JZ","JNZ","JC","JNC"):
                if len(parts) != 2:
                    raise AssemblerError(f"Line {lineno}: {mnem} needs 1 target")
                tgt = parts[1]
                kind, val = self.parse_operand(tgt) if not re.match(r"^[A-Za-z_]\w*$", tgt) else ("LABEL", tgt)
                if kind == "LABEL":
                    if val not in self.labels:
                        raise AssemblerError(f"Line {lineno}: undefined label {val}")
                    addr_val = self.labels[val]
                elif kind == "DIRA":
                    addr_val = val
                else:
                    raise AssemblerError(f"Line {lineno}: jump target must be label or @addr")

                if not (0 <= addr_val < 256):
                    raise AssemblerError(f"Line {lineno}: jump target out of 0..255")

                # Encode full 8-bit address using: addr = (rd_low2<<6) | operand6
                rd_hi = (addr_val >> 6) & 0x03
                op6   = addr_val & 0x3F
                w = self.encode(opcode, rd_hi, MODE_DIR, op6)  # mode ignored by CPU for jumps, but keep MODE_DIR visually
                words.append(w)
                disasm_map[addr] = f"{mnem} {tgt}"
                addr += 1
                continue

            # Two-operand group
            if len(parts) != 3:
                raise AssemblerError(f"Line {lineno}: expected format '{mnem} R?, operand'")
            rd_s = parts[1].upper()
            if rd_s not in REG_NAMES or REG_NAMES[rd_s] > 5:
                raise AssemblerError(f"Line {lineno}: bad dest register {rd_s}")
            rd = REG_NAMES[rd_s]

            op_s = parts[2]
            kind, val = self.parse_operand(op_s)

            # Restrictions: memory is only for LD/ST (RISC-like)
            if mnem not in ("LD","ST") and kind in ("DIRA","LABEL","RIND"):
                raise AssemblerError(f"Line {lineno}: {mnem} cannot use memory operand (use LD first).")

            if mnem == "ST":
                # ST rD, [Rk] or ST rD, @addr or ST rD, label
                if kind == "RIND":
                    w = self.encode(opcode, rd, MODE_RIND, val & 0x07)
                elif kind == "DIRA":
                    if not (0 <= val < 64):
                        raise AssemblerError(f"Line {lineno}: ST direct @addr supports 0..63 in this encoding. Use [Rk] for full RAM.")
                    w = self.encode(opcode, rd, MODE_DIR, val & 0x3F)
                elif kind == "LABEL":
                    if val not in self.labels:
                        raise AssemblerError(f"Line {lineno}: undefined label {val}")
                    a = self.labels[val]
                    if not (0 <= a < 64):
                        raise AssemblerError(f"Line {lineno}: ST label address must be 0..63 (or store via [Rk]).")
                    w = self.encode(opcode, rd, MODE_DIR, a & 0x3F)
                else:
                    raise AssemblerError(f"Line {lineno}: ST needs @addr, label, or [Rk]")
                words.append(w)
                disasm_map[addr] = f"ST {rd_s}, {op_s}"
                addr += 1
                continue

            # LD: can use IMM, REG, DIR (0..63), RIND
            if mnem == "LD":
                if kind == "IMM":
                    imm = val
                    if not (-32 <= imm <= 31):
                        raise AssemblerError(f"Line {lineno}: immediate out of range -32..31 for 6-bit IMM")
                    w = self.encode(opcode, rd, MODE_IMM, imm & 0x3F)
                elif kind == "REG":
                    w = self.encode(opcode, rd, MODE_REG, val & 0x07)
                elif kind == "RIND":
                    w = self.encode(opcode, rd, MODE_RIND, val & 0x07)
                elif kind == "DIRA":
                    if not (0 <= val < 64):
                        raise AssemblerError(f"Line {lineno}: LD direct @addr supports 0..63 in this encoding. Use [Rk] for full RAM.")
                    w = self.encode(opcode, rd, MODE_DIR, val & 0x3F)
                elif kind == "LABEL":
                    if val not in self.labels:
                        raise AssemblerError(f"Line {lineno}: undefined label {val}")
                    a = self.labels[val]
                    if not (0 <= a < 64):
                        raise AssemblerError(f"Line {lineno}: LD label address must be 0..63 (or load via [Rk]).")
                    w = self.encode(opcode, rd, MODE_DIR, a & 0x3F)
                else:
                    raise AssemblerError(f"Line {lineno}: bad operand for LD")
                words.append(w)
                disasm_map[addr] = f"LD {rd_s}, {op_s}"
                addr += 1
                continue

            # ALU ops (REG or IMM only)
            if kind == "IMM":
                imm = val
                if not (-32 <= imm <= 31):
                    raise AssemblerError(f"Line {lineno}: immediate out of range -32..31")
                w = self.encode(opcode, rd, MODE_IMM, imm & 0x3F)
            elif kind == "REG":
                w = self.encode(opcode, rd, MODE_REG, val & 0x07)
            else:
                raise AssemblerError(f"Line {lineno}: {mnem} supports only #imm or Rk")
            words.append(w)
            disasm_map[addr] = f"{mnem} {rd_s}, {op_s}"
            addr += 1

        return origin, words, disasm_map

# ---------------- GUI ----------------
cpu = CPU(mem_words=256)
asm = Assembler()
disasm_map = {}  # addr -> normalized asm line

def format_flags():
    return f"Z={cpu.Z} N={cpu.N} C={cpu.C} V={cpu.V}"

def decode_pretty(d: DecodedIR):
    op_name = None
    for k,v in OP.items():
        if v == d.opcode:
            op_name = k
            break
    if op_name is None:
        op_name = f"OP{d.opcode}"
    return {
        "IR(hex)": f"0x{d.word:04X}",
        "IR(bin)": f"{d.word:016b}",
        "OPCODE": f"{d.opcode} ({op_name})",
        "rD": f"{d.rd}",
        "MODE": f"{d.mode}",
        "OPERAND": f"{d.operand} (0x{d.operand:02X})",
    }

def render_state():
    # Registers
    regs = []
    for i in range(6):
        name = "ACC" if i==0 else f"R{i}"
        regs.append((name, f"0x{cpu.R[i]:04X}", cpu.R[i]))
    reg_lines = "\n".join([f"{n:>3} = {hx} ({dec})" for n,hx,dec in regs])

    # PC/IR/FLAGS
    d = cpu.last_decoded or cpu.fetch_decode() if False else cpu.last_decoded
    ir_fields = ""
    asm_line = ""
    if cpu.last_decoded:
        fields = decode_pretty(cpu.last_decoded)
        ir_fields = "\n".join([f"{k:>10}: {v}" for k,v in fields.items()])
        asm_line = cpu.last_asm if cpu.last_asm else disasm_map.get((cpu.PC-1)&0xFF, "")

    status = (
        f"PC = 0x{cpu.PC:02X} ({cpu.PC})\n"
        f"FLAGS: {format_flags()}\n"
        f"HALTED: {cpu.halted}\n"
    )
    return reg_lines, status, asm_line, ir_fields

def render_memory(start=0, count=32):
    start = max(0, min(255, int(start)))
    count = max(1, min(128, int(count)))
    end = min(256, start + count)
    lines = []
    for a in range(start, end):
        w = cpu.RAM[a]
        mark = " <PC" if a == cpu.PC else ""
        maybe_asm = disasm_map.get(a, "")
        if maybe_asm and not maybe_asm.startswith(".WORD"):
            lines.append(f"{a:03d} 0x{w:04X}  {maybe_asm}{mark}")
        else:
            lines.append(f"{a:03d} 0x{w:04X}  {maybe_asm}{mark}".rstrip())
    return "\n".join(lines)

# Widgets
editor = widgets.Textarea(
    value="",
    placeholder="Type assembly here...",
    description="ASM:",
    layout=widgets.Layout(width="100%", height="260px"),
)

btn_assemble = widgets.Button(description="Assemble & Load", button_style="success")
btn_reset    = widgets.Button(description="Reset", button_style="")
btn_step     = widgets.Button(description="Step", button_style="info")
btn_run      = widgets.ToggleButton(description="Run", button_style="warning", value=False)
speed        = widgets.IntSlider(value=5, min=1, max=20, description="Speed", continuous_update=False)  # higher = faster
mem_start    = widgets.IntText(value=0, description="Mem from")
mem_count    = widgets.IntText(value=48, description="Count")

out = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="8px"))
log = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="8px", height="140px", overflow_y="auto"))

def refresh():
    with out:
        clear_output(wait=True)
        reg_lines, status, asm_line, ir_fields = render_state()
        mem_text = render_memory(mem_start.value, mem_count.value)
        print("=== REGISTERS ===")
        print(reg_lines)
        print("\n=== CPU STATUS ===")
        print(status)
        print("=== IR / DECODE ===")
        if cpu.last_decoded:
            print(f"ASM: {asm_line}")
            print(ir_fields)
        else:
            print("(no instruction executed yet)")
        print("\n=== MEMORY VIEW ===")
        print(mem_text)

def load_sample_1():
    # Sum array elements (N 6..15). Places result into RES_SUM (address must be <64 due to direct LD/ST rules)
    return """; ===== Sample 1: Sum of array elements =====
.ORG 0

LD   R1, #13
LD   R2, [R1]       ; R2 = N
INC  R1             ; R1 -> A1
LD   R3, #0         ; SUM = 0

LOOP_SUM:
CMP  R2, #0
JZ   END_SUM

LD   ACC, [R1]
ADD  R3, ACC
INC  R1
DEC  R2
JMP  LOOP_SUM

END_SUM:
ST   R3, RES_SUM
HLT

; ---- data ----
A:
.WORD 8
.WORD 5, 2, 7, 1, 9, 3, 4, 6

RES_SUM:
.WORD 0
"""

def load_sample_2():
    # Dot product of two arrays of 6 unsigned elements. Result into RES_DOT (<64)
    return """; ===== Sample 2: Dot product (convolution) of two arrays =====
.ORG 0

LD   R1, A_PTR       ; загрузили содержимое ячейки A_PTR (это адрес массива A)
LD   R1, [R1]        ; R1 = address of A
LD   R3, [R1]        ; R3 = N
INC  R1              ; -> A1

LD   R2, #B_PTR
LD   R2, [R2]        ; R2 = address of B
INC  R2              ; -> B1

LD   R4, #0          ; S = 0

LOOP_DOT:
CMP  R3, #0
JZ   END_DOT

LD   ACC, [R1]       ; ACC = A[i]
LD   R5, [R2]        ; R5  = B[i]
MUL  ACC, R5         ; ACC = A[i]*B[i] (low16)
ADD  R4, ACC         ; S += product

INC  R1
INC  R2
DEC  R3
JMP  LOOP_DOT

END_DOT:
ST   R4, RES_DOT
HLT

; ---- data area (labels <64) ----
A_PTR:
.WORD 100
B_PTR:
.WORD 120

RES_DOT:
.WORD 0

; ---- arrays ----
.ORG 100
A:
.WORD 6
.WORD 2, 3, 4, 5, 6, 7

.ORG 120
B:
.WORD 6
.WORD 10, 20, 30, 40, 50, 60
"""

dropdown_samples = widgets.Dropdown(
    options=[("— choose sample —", ""), ("Task 1: Sum array", "s1"), ("Task 2: Dot product", "s2")],
    description="Samples",
    value=""
)

def on_choose_sample(change):
    if change["new"] == "s1":
        editor.value = load_sample_1()
    elif change["new"] == "s2":
        editor.value = load_sample_2()

dropdown_samples.observe(on_choose_sample, names="value")

run_thread = None
run_stop = threading.Event()

def log_print(msg):
    with log:
        print(msg)

def do_reset(_=None):
    global disasm_map
    cpu.reset()
    disasm_map = {}
    log.clear_output()
    refresh()

def do_assemble(_=None):
    global disasm_map
    src = editor.value
    try:
        origin, words, dmap = asm.assemble(src)
        cpu.reset()
        cpu.load_words(origin, words)
        disasm_map = dmap
        log.clear_output()
        log_print(f"Assembled OK. Origin={origin}, words={len(words)}")
    except Exception as e:
        log_print(f"ASSEMBLE ERROR: {e}")
    refresh()

def do_step(_=None):
    if cpu.halted:
        log_print("CPU is halted.")
        refresh()
        return
    # current instruction disasm line from map at PC
    asm_line = disasm_map.get(cpu.PC, "")
    try:
        cpu.step(disasm_line=asm_line)
    except Exception as e:
        log_print(f"RUNTIME ERROR: {e}")
        cpu.halted = True
    refresh()

def runner():
    # run loop
    while not run_stop.is_set():
        if cpu.halted:
            break
        do_step()
        # speed slider: higher -> faster
        delay = max(0.02, 0.35 - (speed.value * 0.015))
        time.sleep(delay)
    btn_run.value = False

def on_toggle_run(change):
    global run_thread
    if change["new"] is True:
        if cpu.halted:
            log_print("CPU is halted. Reset or assemble again.")
            btn_run.value = False
            return
        run_stop.clear()
        run_thread = threading.Thread(target=runner, daemon=True)
        run_thread.start()
    else:
        run_stop.set()

def on_mem_change(change):
    refresh()

btn_reset.on_click(do_reset)
btn_assemble.on_click(do_assemble)
btn_step.on_click(do_step)
btn_run.observe(on_toggle_run, names="value")
mem_start.observe(on_mem_change, names="value")
mem_count.observe(on_mem_change, names="value")

# Layout
controls = widgets.HBox([btn_assemble, btn_reset, btn_step, btn_run, speed])
mem_controls = widgets.HBox([mem_start, mem_count, dropdown_samples])
ui = widgets.VBox([
    mem_controls,
    editor,
    controls,
    widgets.HTML("<b>CPU State</b>"),
    out,
    widgets.HTML("<b>Build/Runtime Log</b>"),
    log
])

display(ui)
do_reset()
